In [7]:
!pip install -q langchain langchain-community langchain-huggingface chromadb langchain-chroma sentence-transformers

In [ ]:
pip install langchain-core

In [11]:
import json
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# 1. Load your JSON data
with open("/content/traite_v5_docs.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# 2. Build the list of Document objects
docs = []
for item in data:
    text = f"Content:{item['Content']}".strip()
    docs.append(Document(
        page_content=text,
        metadata={
            "source": item.get("source", ""),
            "id": item.get("id", "")
        }
    ))

# 3. Initialize HuggingFaceEmbeddings (using model_name)
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

# 4. Batch ingestion into Chroma
batch_size = 100
db = None

for i in range(0, len(docs), batch_size):
    print(f"Processing {i} -> {i + batch_size}")
    batch = docs[i:i + batch_size]

    if db is None:
        # Initialize the database with the first batch
        db = Chroma.from_documents(
            documents=batch,
            embedding=embeddings,
            persist_directory="db_v2"
        )
    else:
        # Incrementally add subsequent batches to the existing database
        db.add_documents(documents=batch)

/tmp/ipykernel_611/174647925.py:23: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Processing 0 -> 100
Processing 100 -> 200
Processing 200 -> 300
Processing 300 -> 400
Processing 400 -> 500


In [18]:
!zip -r db_v2 /content/db_v2

  adding: content/db_v2/ (stored 0%)
  adding: content/db_v2/chroma.sqlite3 (deflated 53%)
  adding: content/db_v2/07740090-d665-4814-aa07-c7cd1b01960e/ (stored 0%)
  adding: content/db_v2/07740090-d665-4814-aa07-c7cd1b01960e/link_lists.bin (stored 0%)
  adding: content/db_v2/07740090-d665-4814-aa07-c7cd1b01960e/data_level0.bin (deflated 100%)
  adding: content/db_v2/07740090-d665-4814-aa07-c7cd1b01960e/length.bin (deflated 67%)
  adding: content/db_v2/07740090-d665-4814-aa07-c7cd1b01960e/header.bin (deflated 63%)


In [19]:
from google.colab import files
files.download("/content/db_v2.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>